<center style="padding: 1em 2em;  margin: 1em 0;">
    <h1 style="font-size: 2.2em; margin: 0 0 0.5em 0; color:rgb(52, 72, 92);">
        «Вложенные функции, замыкания, декораторы»
    </h1>
</center>


На прошлой паре имя искалось в своём кадре, потом в модуле, потом среди встроенных. Кадр умирал на `return`, локальные имена уходили вместе с ним.

Между своим кадром и модулем в `LEGB` остался шаг, который мы ещё не видели: кадр функции, *внутри* которой объявлена текущая.

Сегодня этот шаг станет живым. Внутренняя функция увидит имена, которые взяла, даже когда внешний вызов уже кончился. Оттуда же получится обёртка вокруг функции.

---


## **Сегодня пройдемся по этому плану:**

1. **Вложенная функция** – читает имена внешней.
2. **Память после `return`** – почему модуль сюда не годится, две независимые памяти.
3. **Декоратор** – функция, которая принимает функцию и возвращает другую.

---


## **Функция внутри функции.**

Мы уже передаём функцию в функцию и кладём её в другое имя. Не хватает случая, когда *создать* функцию нужно там, где снаружи уже лежат нужные имена.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Вложенная функция:</strong><br>функция, объявленная в теле другой. Снаружи по своему имени она не видна: это имя живёт в кадре внешней.
</div>

<img src="../../Other/images/python_namespace.png" alt="Пространства имён LEGB" width="420">

На прошлой паре внутреннее кольцо было «свой кадр». Сегодня в нём появляется ещё один слой: кадр *другой* функции, внутри которой мы объявлены.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Enclosing:</strong><br>кадр функции, внутри которой объявлена текущая. Ближе, чем модуль, дальше, чем свой кадр.
</div>


In [ ]:
def outer():
    label = "enclosing"

    def inner():
        print(label)

    inner()


outer()  # -> enclosing
# inner()  # NameError: имени inner снаружи нет

enclosing


Своего `label` у `inner` нет – шаг E в `LEGB`. Пока `inner` вызывается до `return` из `outer`, снаружи ещё живой кадр.

Чтение уже работает. Запись в то же имя – та же ловушка, что с `counter` в модуле: Python считает имя локальным на весь кадр и до внешней функции не доходит.


In [4]:
def make_counter():
    count = 0

    def bump():
        count = count + 1  # UnboundLocalError: локальный count ещё не создан
        return count

    return bump


make_counter()()

UnboundLocalError: local variable 'count' referenced before assignment

## **Зачем помнить после return.**

Хотим функцию, которая *после* конца внешнего вызова всё ещё знает своё число: сколько раз её нажали. Это число не аргумент каждого вызова. И не имя модуля: модуль один на всех.

Сначала положим счётчик в модуль – туда мы уже умеем писать из функции.


In [ ]:
count = 0


def make_counter():
    

    def bump():
        global count
        count = count + 1
        return count

    return bump


clicks = make_counter()
taps = make_counter()
print(clicks())  # -> 1
print(taps())    # -> 2
print(clicks())  # -> 3

Два объекта `bump`, одно имя `count`. `taps` продолжает чужой счёт.

Чтобы писать во внешнюю функцию, это говорят явно: `nonlocal`.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong><code>nonlocal</code>:</strong><br>объявление: запись идёт в ближайшее enclosing-имя, не в свой кадр и не в модуль.
</div>


In [ ]:
def make_counter():
    count = 0

    def bump(a3):
        nonlocal count
        count = count + 1
        return count

    return bump


clicks = make_counter()
taps = make_counter()


print(clicks())  # -> 1
print(clicks())  # -> 2
print(taps())    # -> 1
print(taps())    # -> 1

print(clicks())  # -> 2

SyntaxError: no binding for nonlocal 'count' found (1157592508.py, line 6)

`make_counter` уже вернулась. Этого вызова на стеке больше нет.

Обычные локальные имена ушли бы вместе с кадром. `count` не ушёл: объект `bump` всё ещё держит на него ссылку.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Замыкание:</strong><br>функция вместе с именами, которые она взяла снаружи. Внешний вызов закончился, внутренняя эти имена всё ещё видит.
</div>

Два вызова `make_counter()` – два отдельных запомненных состояния. `clicks` не видит `count` от `taps`.

---


## **Одно имя на всех: цикл.**

Замыкание держит *имя*, не снимок значения в момент `def`. Цикл кадр не создаёт: три функции из *одного* вызова смотрят на одно и то же `factor`. После цикла в нём последнее значение.


In [ ]:
def make_broken():
    funcs = []
    for factor in (2, 3, 4):
        def multiply(value):
            return value * factor

        funcs.append(multiply)
    return funcs


funcs = make_broken()
print([f(10) for f in funcs])  # -> [40, 40, 40]

Своё состояние на каждый множитель – свой вызов внешней функции.


In [ ]:
def make_multiplier(factor):
    def multiply(value):
        return value * factor

    return multiply


funcs = [make_multiplier(factor) for factor in (2, 3, 4)]
print([f(10) for f in funcs])  # -> [20, 30, 40]

Короткий обход: `def multiply(value, factor=factor)` закрепит текущее значение – дефолт считается на `def`.

---


## **Декоратор.**

Хотим добавить поведение вокруг функции, не трогая тело и все места вызова.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Декоратор:</strong><br>функция, которая принимает функцию и возвращает новую с тем же способом вызова.
</div>

Сначала без `@`. Имя `add` начинает указывать на другую функцию. Новая помнит старую – то же замыкание, только захваченное имя само является функцией.


In [17]:
def logger(func):

    def wrapper(*args, **kwargs):
        print("вызываем", func.__name__)
        result = func(*args, **kwargs)
        print("готово")
        return result

    return wrapper


# def add(a, b):
#     return a + b

def aboba(a):
    return a * 4

aboba = logger(aboba)

print(aboba(2))
# -> вызываем add
# -> готово
# -> 5

вызываем aboba
готово
8


In [ ]:
@aboba
@logger
def add(a, b):
    return a + b

add = aboba(logger(add))

Это ровно `add = logger(add)` после объявления.


In [ ]:
@logger
def mul(a, b):
    return a * b


print(mul(3, 4))
# -> вызываем mul
# -> готово
# -> 12

### **Цепочка декораторов.**

Несколько строк `@` – несколько обёрток подряд. Ближний к `def` применяется первым.


In [ ]:
def outer(func):
    def wrapper(*args, **kwargs):
        print("outer")
        return func(*args, **kwargs)

    return wrapper


def inner(func):
    def wrapper(*args, **kwargs):
        print("inner")
        return func(*args, **kwargs)

    return wrapper


@outer
@inner
def ping():
    print("пинг")


ping()
# -> outer
# -> inner
# -> пинг

Это `ping = outer(inner(ping))`. На вызове сначала внешняя обёртка.

### **Имя и документация.**

У функции есть `__name__` и `__doc__`. Их читают `help` и IDE.

Сделаем декоратор, который при вызове пишет, что функция устарела. Тело `show` не трогаем.


In [22]:
def deprecated(func):

    def wrapper(*args, **kwargs):
        print(func.__name__, "is deprecated!")
        return func(*args, **kwargs)

    return wrapper


@deprecated
def show(x):
    'This is a really nice looking docstring'
    print(x)


print(show(1))
# -> show is deprecated!
# -> 1

show is deprecated!
1
None


В сообщении имя ещё настоящее: `func` – старая функция. А имя `show` уже указывает на `wrapper`.


In [ ]:
print(show.__name__)
print(show.__doc__)
# -> wrapper
# -> None

`help(show)` расскажет про обёртку, не про `show`.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong><code>functools.wraps(func)</code>:</strong><br>декоратор на обёртку. Копирует имя и документацию исходной функции.
</div>


In [23]:
from functools import wraps


def deprecated(func):
    
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(func.__name__, "is deprecated!")
        return func(*args, **kwargs)

    return wrapper


@deprecated
def show(x):
    'This is a really nice looking docstring'
    print(x)


print(show.__name__)
print(show.__doc__)
# -> show
# -> This is a really nice looking docstring

show
This is a really nice looking docstring


### **Состояние в обёртке.**

Замыкание внутри декоратора умеет помнить, вызывали ли функцию. Первый вызов считает результат, следующие отдают сохранённое.


In [ ]:
def once(func):
    called = False
    saved = None

    @wraps(func)
    def wrapper(*args, **kwargs):
        nonlocal called, saved
        if not called:
            saved = func(*args, **kwargs)
            called = True
        return saved

    return wrapper


@once
def load_config():
    print("читаем конфиг")
    return {"debug": True}


print(load_config())
# -> читаем конфиг
# -> {'debug': True}
print(load_config())  # -> {'debug': True}

Повторные вызовы аргументы не смотрят: это не кэш по входу, а «посчитали один раз».

Тот же приём, другой вопрос: запомнить момент *первого* вызова и от него мерить всё остальное. Две функции под одним декоратором – два старта, как два вызова `make_counter()`.

`time.perf_counter()` – часы для интервалов: разница двух показаний, не дата.


In [24]:
import time


def since_first_call(func):
    started_at = None

    @wraps(func)
    def wrapper(*args, **kwargs):
        nonlocal started_at
        now = time.perf_counter()
        if started_at is None:
            started_at = now
        print(func.__name__, "от старта", round(now - started_at, 2))
        return func(*args, **kwargs)

    return wrapper


@since_first_call
def load_users():
    return ["anna"]


@since_first_call
def load_orders():
    return [1]


print(load_users())
# -> load_users от старта 0.0
# -> ['anna']
time.sleep(0.2)
print(load_users())  # то же имя, уже не ноль, затем ['anna']
print(load_orders())
# -> load_orders от старта 0.0
# -> [1]
time.sleep(0.2)
print(load_users())  # users живёт дальше, orders его не сбросил

load_users от старта 0.0
['anna']
load_users от старта 0.2
['anna']
load_orders от старта 0.0
[1]
load_users от старта 0.41
['anna']


In [30]:
print(load_orders())

load_orders от старта 53.34
[1]


Смотрите числа `от старта`: у `load_users` они растут; у `load_orders` первый вызов снова ноль. Модульный `started_at` снова склеил бы оба отсчёта.

### **Декоратор с параметром.**

`logger` сам был декоратором: принял функцию, вернул функцию. Если перед этим нужен параметр – «сколько раз повторить», – появляется ещё одна функция снаружи.

`@repeat(3)` сначала вызывает `repeat(3)`. Наружу выходит декоратор, и уже он оборачивает `beep`.


In [ ]:
def deco_outer(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = func(*args, **kwargs)
            return result

        return wrapper


@deco_outer(3)
def beep():
    print("beep")


beep()
# -> beep
# -> beep
# -> beep

In [ ]:
def repeat(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = func(*args, **kwargs)
            return result

        return wrapper

    return decorator


@repeat(3)
def beep():
    print("beep")


beep()
# -> beep
# -> beep
# -> beep

## **Дополнительно почитать.**

Не обязательно до следующей пары.

- [Execution model](https://docs.python.org/3/reference/executionmodel.html) – официально, как ищутся имена: свой блок, enclosing, `global`, `nonlocal`.
- [First-class](https://docs.python.org/3/glossary.html#term-first-class) – функция как объект: можно передать, вернуть, положить в имя.
- [functools.wraps](https://docs.python.org/3/library/functools.html#functools.wraps) – что именно копируется на обёртку.

Внешний вызов заканчивается на `return`. Следующий вопрос: можно ли оставить кадр живым и отдавать из него значения по одному.
